In [307]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.model_selection import train_test_split  
from sklearn.model_selection import (
    StratifiedKFold, 
    cross_val_score, 
    cross_validate
)
from sklearn.model_selection import RandomizedSearchCV

from sklearn.linear_model import LogisticRegression  
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


from sklearn.metrics import accuracy_score  


In [308]:
RANDOM_STATE = 777

In [309]:
df = pd.read_csv(
    filepath_or_buffer="gaming_and_mental_health.csv"
)

In [310]:
df.head()

,record_id,age,gender,daily_gaming_hours,game_genre,primary_game,gaming_platform,sleep_hours,sleep_quality,sleep_disruption_frequency,...,continued_despite_problems,eye_strain,back_neck_pain,weight_change_kg,exercise_hours_weekly,social_isolation_score,face_to_face_social_hours_weekly,monthly_game_spending_usd,years_gaming,gaming_addiction_risk_level
0,GD0001,17,Male,11.1,Mobile Games,Clash of Clans,PC,3.7,Very Poor,Sometimes,...,True,True,False,6.8,3.7,7,1.3,383.70,3,Severe
1,GD0002,21,Male,3.0,MOBA,Dota 2,PC,7.2,Fair,Rarely,...,False,False,False,0.4,8.5,2,10.7,46.64,1,Low
2,GD0003,23,Male,7.6,FPS,CS:GO,Multi-platform,4.4,Fair,Often,...,True,False,True,1.8,7.1,5,3.2,100.81,6,Severe
3,GD0004,20,Female,7.2,RPG,Skyrim,Multi-platform,5.1,Fair,Often,...,False,True,True,0.2,5.2,4,9.1,51.60,7,High
4,GD0005,18,Male,6.8,Battle Royale,Apex Legends,PC,3.4,Poor,Never,...,False,False,False,0.5,6.1,4,4.5,32.57,1,Moderate


In [311]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 27 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   record_id                         1000 non-null   object 
 1   age                               1000 non-null   int64  
 2   gender                            1000 non-null   object 
 3   daily_gaming_hours                1000 non-null   float64
 4   game_genre                        1000 non-null   object 
 5   primary_game                      1000 non-null   object 
 6   gaming_platform                   1000 non-null   object 
 7   sleep_hours                       1000 non-null   float64
 8   sleep_quality                     1000 non-null   object 
 9   sleep_disruption_frequency        1000 non-null   object 
 10  academic_work_performance         1000 non-null   object 
 11  grades_gpa                        754 non-null    float64
 12  work_pr

### Data Preparation

In [312]:
df["gaming_addiction_risk_level"].unique()

array(['Severe', 'Low', 'High', 'Moderate'], dtype=object)

In [313]:
df["record_id"].nunique()

1000

In [314]:
df = df.drop(columns=["record_id"])

In [315]:
columns_to_hot_encode = [
    "gender",
    "game_genre", 
    "primary_game", 
    "gaming_platform",
    "mood_state",
]

df = pd.get_dummies(data=df, columns=columns_to_hot_encode)

In [316]:
sleep_quality_map = {
    "Insomnia": 0,
    "Very Poor": 1,
    "Poor": 2,
    "Fair": 3,
    "Good": 4
}

df["sleep_quality"] = df["sleep_quality"].map(arg=sleep_quality_map)

In [317]:
df["mood_swing_frequency"].unique()

array(['Never', 'Often', 'Rarely', 'Daily', 'Sometimes'], dtype=object)

In [318]:
mood_swing_freq_map = {
    "Never": 0,
    "Rarely": 1,
    "Sometimes": 2,
    "Often": 3,
    "Daily": 4
}

df["mood_swing_frequency"] = df["mood_swing_frequency"].map(mood_swing_freq_map)

In [319]:
academic_perf_map = {
    "Failing": 0,
    "Poor": 1,
    "Below Average": 2,
    "Average": 3,
    "Good": 4,
    "Excellent": 5
}

df["academic_work_performance"] = df["academic_work_performance"].map(academic_perf_map)

In [320]:
df["sleep_disruption_frequency"].unique()

array(['Sometimes', 'Rarely', 'Often', 'Never', 'Always'], dtype=object)

In [321]:
sleep_disruption_map = {
    "Never": 0,
    "Rarely": 1,
    "Sometimes": 2,
    "Often": 3,
    "Always": 4
}

df["sleep_disruption_frequency"] = df["sleep_disruption_frequency"].map(sleep_disruption_map)

In [322]:
boolean_columns = df.select_dtypes(include="bool").columns
df[boolean_columns] = df[boolean_columns].astype(int)

In [323]:
addiction_map = {
    "Low": 0,
    "Moderate": 1,
    "High": 2,
    "Severe": 3
}

df["gaming_addiction_risk_level"] = df["gaming_addiction_risk_level"].map(addiction_map)

In [324]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 68 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   age                                1000 non-null   int64  
 1   daily_gaming_hours                 1000 non-null   float64
 2   sleep_hours                        1000 non-null   float64
 3   sleep_quality                      1000 non-null   int64  
 4   sleep_disruption_frequency         1000 non-null   int64  
 5   academic_work_performance          1000 non-null   int64  
 6   grades_gpa                         754 non-null    float64
 7   work_productivity_score            674 non-null    float64
 8   mood_swing_frequency               1000 non-null   int64  
 9   withdrawal_symptoms                1000 non-null   int64  
 10  loss_of_other_interests            1000 non-null   int64  
 11  continued_despite_problems         1000 non-null   int64 

#### Handling NULL Values

In [325]:
# NOTE: TEMPORARY
df["grades_gpa"] = df["grades_gpa"].fillna(df["grades_gpa"].median())
df["work_productivity_score"] = (
    df["work_productivity_score"]
    .fillna(df["work_productivity_score"].median())
)

#### Splitting and Scaling

In [326]:
y = df["gaming_addiction_risk_level"]
X = df.drop(columns=["gaming_addiction_risk_level"])

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=0.2, random_state=RANDOM_STATE  
) 

In [327]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Logistic Regression

#### Simple Logistic Regression

In [328]:
logistic_regression = LogisticRegression(
    C=0.7,
    solver="lbfgs",
    max_iter=5000,
    l1_ratio=0
)

logistic_regression.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.7
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [329]:
y_pred = logistic_regression.predict(X_test_scaled)

# Evaluation  
logistic_regression_accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", logistic_regression_accuracy) 

Accuracy: 0.92


In [330]:
coefficients = logistic_regression.coef_[0]
odds_ratios = np.exp(coefficients)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': coefficients,
    'odds_ratio': odds_ratios
})

feature_importance.sort_values(by='coefficient', ascending=False)

,feature,coefficient,odds_ratio
17,face_to_face_social_hours_weekly,0.840375,2.317236
2,sleep_hours,0.690322,1.994357
15,exercise_hours_weekly,0.409861,1.506608
3,sleep_quality,0.208737,1.232121
48,primary_game_PUBG Mobile,0.180558,1.197886
...,...,...,...
18,monthly_game_spending_usd,-1.209464,0.298357
1,daily_gaming_hours,-1.419983,0.241718
11,continued_despite_problems,-1.584647,0.205020
9,withdrawal_symptoms,-3.024024,0.048605


In [331]:
treshold = 0.2

feature_importance = feature_importance[abs(feature_importance["coefficient"]) > treshold]

feature_importance.sort_values(by='coefficient', ascending=False)
important_columns = feature_importance["feature"]

In [332]:
y = df["gaming_addiction_risk_level"]
X = df.drop(columns=["gaming_addiction_risk_level"])

X = X.loc[:, important_columns]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=0.2, random_state=RANDOM_STATE  
) 

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [333]:
logistic_regression = LogisticRegression(
    C=0.7,
    solver="lbfgs",
    max_iter=5000,
    l1_ratio=0
)

logistic_regression.fit(X_train_scaled, y_train)

y_pred = logistic_regression.predict(X_test_scaled)

# Evaluation  
logistic_regression_accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", logistic_regression_accuracy) 

Accuracy: 0.97


#### Cross Validation with Logistic Regression

In [334]:
y = df["gaming_addiction_risk_level"]
X = df.drop(columns=["gaming_addiction_risk_level"])

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=0.2, random_state=RANDOM_STATE  
) 

X_scaled = scaler.fit_transform(X)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [335]:
scoring = ["accuracy", "f1_weighted", "precision_weighted", "recall_weighted"]

logistic_regression = LogisticRegression(
    C=0.7,
    solver="lbfgs",
    max_iter=5000,
    l1_ratio=0
)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(logistic_regression, X_scaled, y, cv=cv, scoring="f1_weighted")

print("Fold accuracies:", scores)
print("Mean F1 Weighted:", np.mean(scores))

Fold accuracies: [0.93910714 0.97957521 0.95042184 0.96088204 0.96       0.96054586
 0.95963801 0.98       0.93982906 0.96040504]
Mean F1 Weighted: 0.959040420181547


In [336]:
results = cross_validate(
    logistic_regression,
    X_scaled,
    y,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    print(metric, results["test_" + metric].mean())

accuracy 0.959
f1_weighted 0.959040420181547
precision_weighted 0.9610947447805032
recall_weighted 0.959


#### Random Search of Hyperparameters

In [337]:
y = df["gaming_addiction_risk_level"]
X = df.drop(columns=["gaming_addiction_risk_level"])

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=0.2, random_state=RANDOM_STATE  
) 

X_scaled = scaler.fit_transform(X)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [338]:
param_grid = {
    "C": np.linspace(start=0.001, stop=1, num=30),
    "solver": ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
    "max_iter": [num for num in range(100, 5000, 250)]
}

rn_logistic_regression = RandomizedSearchCV(
    estimator=LogisticRegression(),
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=RANDOM_STATE,
    n_jobs=-1
)


rn_logistic_regression.fit(X_train_scaled, y_train)

print("Best Parameters:", rn_logistic_regression.best_params_)
print("Best CV Score:", rn_logistic_regression.best_score_)


Best Parameters: {'solver': 'sag', 'max_iter': 3100, 'C': np.float64(0.965551724137931)}
Best CV Score: 0.9325000000000001


In [339]:
scoring = ["accuracy", "f1_weighted", "precision_weighted", "recall_weighted"]

rn_logistic_regression = rn_logistic_regression.best_estimator_
results = cross_validate(
    rn_logistic_regression,
    X_scaled,
    y,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    print(metric, results["test_" + metric].mean())

accuracy 0.9599999999999997
f1_weighted 0.9600766301453154
precision_weighted 0.9621152926150216
recall_weighted 0.9599999999999997


In [340]:
importances = np.abs(rn_logistic_regression.coef_[0])
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

important_features = feature_importance[feature_importance > 0.5].index
X_train_selected = X_train[important_features]
X_test_selected = X_test[important_features]

important_features


loss_of_other_interests           4.081475
withdrawal_symptoms               3.366318
continued_despite_problems        1.789975
daily_gaming_hours                1.572587
monthly_game_spending_usd         1.350181
                                    ...   
primary_game_CS:GO                0.009523
mood_state_Anxious                0.009314
mood_state_Irritable              0.008702
gaming_platform_Multi-platform    0.004675
mood_state_Normal                 0.003132
Length: 67, dtype: float64


Index(['loss_of_other_interests', 'withdrawal_symptoms',
       'continued_despite_problems', 'daily_gaming_hours',
       'monthly_game_spending_usd', 'social_isolation_score',
       'face_to_face_social_hours_weekly', 'sleep_hours'],
      dtype='object')

In [341]:
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)       

logistic_regression = LogisticRegression(
    C=np.float64(0.965551724137931),
    max_iter=3100,
    solver='sag'
)

logistic_regression.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",np.float64(0.965551724137931)
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of

In [342]:
results = cross_validate(
    rn_logistic_regression,
    X_scaled,
    y,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    print(metric, results["test_" + metric].mean())

accuracy 0.9599999999999997
f1_weighted 0.9600766301453154
precision_weighted 0.9621152926150216
recall_weighted 0.9599999999999997


### KNeighbors Classifier

In [343]:
y = df["gaming_addiction_risk_level"]
X = df.drop(columns=["gaming_addiction_risk_level"])

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=0.2, random_state=RANDOM_STATE  
) 

X_scaled = scaler.fit_transform(X)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [344]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="minkowski",
    p=2
)
knn.fit(X_train_scaled, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [345]:
results = cross_validate(
    knn,
    X_scaled,
    y,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    print(metric, results["test_" + metric].mean())

accuracy 0.688
f1_weighted 0.6570794736998234
precision_weighted 0.6564072872908041
recall_weighted 0.688


#### Random Search of Hyperparameters for KNN

In [346]:
param_grid = {
    "n_neighbors": list(range(3, 25)),
    "weights": ["uniform", "distance"],
    "p": [1, 2],
    "algorithm": ['auto', 'ball_tree', 'kd_tree', 'brute']
}

random_knn = RandomizedSearchCV(
    estimator=KNeighborsClassifier(),
    param_distributions=param_grid,
    n_iter=20, # number of random combinations
    cv=5,
    scoring="accuracy",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_knn.fit(X_train_scaled, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsClassifier()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'algorithm': ['auto', 'ball_tree', ...], 'n_neighbors': [3, 4, ...], 'p': [1, 2], 'weights': ['uniform', 'distance']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 

In [347]:
print("Best Parameters (Random):", random_knn.best_params_)

best_knn = random_knn.best_estimator_

Best Parameters (Random): {'weights': 'distance', 'p': 1, 'n_neighbors': 4, 'algorithm': 'brute'}


In [348]:
results = cross_validate(
    random_knn,
    X_scaled,
    y,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    print(metric, results["test_" + metric].mean())

accuracy 0.694
f1_weighted 0.6650430220524461
precision_weighted 0.6619742817675591
recall_weighted 0.694


### Boosting Techniques

#### Gradient Boosting Classifier

In [349]:
y = df["gaming_addiction_risk_level"]
X = df.drop(columns=["gaming_addiction_risk_level"])

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=0.2, random_state=RANDOM_STATE  
) 

In [350]:
gradient_boosting = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

gradient_boosting.fit(X_train, y_train)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",0.8
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``,

In [351]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

In [352]:
results = cross_validate(
    gradient_boosting,
    X,
    y,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    print(metric, results["test_" + metric].mean())

accuracy 0.961
f1_weighted 0.9609063976956674
precision_weighted 0.964005013368984
recall_weighted 0.961


#### Extreme Gradient Boosting

In [353]:
extreme_gb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

extreme_gb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [354]:
results = cross_validate(
    extreme_gb,
    X,
    y,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    print(metric, results["test_" + metric].mean())

accuracy 0.9760000000000002
f1_weighted 0.9760396117341807
precision_weighted 0.9773603641456583
recall_weighted 0.9760000000000002


### Cat Boosting

In [355]:
cat_boosting = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    random_state=42,
    verbose=False
)

cat_boosting.fit(X_train, y_train)

In [356]:
y_pred = cat_boosting.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 1.0
